Evaluate direct, CFM, MFM (?) models in generative dropout exp. To eval a given method:
- Load a surrogate model (there are 20 seeds each)
- Load training data, for the timepoints at t=0 and t=0.505
- Load eval data at times t = 0.101, 0.202, 0.303, 0.404, 0.606, 0.707, 0.808, 0.909
- Generate pushforwards at times t = 0.101, 0.202, 0.303, 0.404, 0.606, 0.707, 0.808, 0.909
- Calculate wasserstein distance for each time, and the avg. wasserstein across times

In [8]:
import torch
import numpy as np
import ot
from torch import nn, Tensor

In [3]:
def compute_wasserstein_distance(samples1, samples2):
    """
    Compute Wasserstein distance between two sets of samples using POT library.
    
    Args:
        samples1: JAX array of shape (n_samples1, dim) or numpy array
        samples2: JAX array of shape (n_samples2, dim) or numpy array
        
    Returns:
        float: The Wasserstein distance
    """
    # Convert from JAX arrays to numpy for POT library compatibility
    samples1_np = np.array(samples1)
    samples2_np = np.array(samples2)
    

    M = ot.dist(samples1_np, samples2_np)
    a = np.ones(samples1_np.shape[0]) / samples1_np.shape[0]  # uniform weights
    b = np.ones(samples2_np.shape[0]) / samples2_np.shape[0]  # uniform weights
        
    return float(ot.emd2(a, b, M))

In [9]:
dropout_data = torch.load('../NLOT/data/diffusion_2moons_dropout.pt')
t0_samples = dropout_data[0]
t05_samples = dropout_data[5]

eval_samples = dropout_data[[1,2,3,4,6,7,8,9]]
index_to_eval_time = {0: 0.101, 1: 0.202, 2: 0.303, 3: 0.404, 4: 0.505, 5: 0.707, 6: 0.808, 7: 0.909}

In [12]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [50]:
def print_results(results, name):
    mean = np.mean(results)
    std = np.std(results)
    ci = 1.96 * std / np.sqrt(len(results))
    print(f"{name}: {mean:.3f} ± {ci:.3f}")

# Direct method

In [10]:
dropout_data.shape

torch.Size([11, 1000, 3])

In [11]:
class Swish(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x: Tensor) -> Tensor: 
        return torch.sigmoid(x) * x

# Model class with FiLM conditioning
class FiLMMLP(nn.Module):
    def __init__(self, input_dim: int = 2, time_dim: int = 1, hidden_dim: int = 128, cond_dim: int = 1):
        super().__init__()
        
        self.input_dim = input_dim
        self.time_dim = time_dim
        self.hidden_dim = hidden_dim
        self.cond_dim = cond_dim
        
        # Main network layers
        self.input_layer = nn.Linear(input_dim + time_dim, hidden_dim)
        self.hidden_layers = nn.ModuleList([
            nn.Linear(hidden_dim, hidden_dim) for _ in range(3)
        ])
        self.output_layer = nn.Linear(hidden_dim, input_dim)
        
        self.activation = Swish()
        
        # FiLM generator network
        # It will generate gammas and betas for the 4 layers (input + 3 hidden)
        num_film_layers = len(self.hidden_layers) + 1 
        self.film_generator = nn.Sequential(
            nn.Linear(cond_dim, hidden_dim),
            self.activation,
            nn.Linear(hidden_dim, num_film_layers * hidden_dim * 2) # *2 for gamma and beta
        )
        
    def forward(self, x: Tensor, t: Tensor, cond: Tensor) -> Tensor:
        sz = x.size()
        x = x.reshape(-1, self.input_dim)
        t = t.reshape(-1, self.time_dim).float()
        cond = cond.reshape(-1, self.cond_dim).float()

        # Expand time to match batch size
        t = t.reshape(-1, 1).expand(x.shape[0], 1)
        
        # Generate FiLM parameters from condition
        film_params = self.film_generator(cond)
        
        # Split FiLM params into gammas and betas for each layer
        num_film_layers = len(self.hidden_layers) + 1
        gammas = film_params[:, :num_film_layers * self.hidden_dim].reshape(-1, num_film_layers, self.hidden_dim)
        betas = film_params[:, num_film_layers * self.hidden_dim:].reshape(-1, num_film_layers, self.hidden_dim)

        # Main forward pass
        h = torch.cat([x, t], dim=1)
        
        # Input layer
        h = self.input_layer(h)
        h = gammas[:, 0, :] * h + betas[:, 0, :] # Apply FiLM
        h = self.activation(h)
        
        # Hidden layers
        for i, layer in enumerate(self.hidden_layers):
            h = layer(h)
            h = gammas[:, i + 1, :] * h + betas[:, i + 1, :] # Apply FiLM
            h = self.activation(h)
            
        # Output layer
        output = self.output_layer(h)
        
        return output.reshape(*sz)

In [13]:
def naive_forward(sample, cond, lambda_val, naive_model):
    """
    Uses the trained naive model to push samples forward to target lambda, where naive model can take in any lambda for a given sample, condition pair.
    """
    if lambda_val <= 0:
        return sample
    naive_sample = naive_model(torch.tensor(sample, device=device), torch.tensor(lambda_val, device=device), torch.tensor(cond, device=device)).cpu().detach().numpy()

    #clip to range [-4,4]
    naive_sample[0][0] = np.clip(naive_sample[0][0], -4, 4)
    naive_sample[0][1] = np.clip(naive_sample[0][1], -4, 4)
    
    return naive_sample

In [49]:
t0_ambient = t0_samples[:,:2]
t0_condition = t0_samples[:,2:]

avg_wass_dists_naive = []

for i in range(20):
    naive_model = FiLMMLP(input_dim=2, time_dim=1, hidden_dim=64, cond_dim=1).to(device)
    naive_model.load_state_dict(torch.load(f'../rebuttal_baselines/naive_dropout_FiLM_model_{i}.pt', map_location=device))

    wass_dists_model_i = []

    for time_index, eval_time in index_to_eval_time.items():
        eval_sample = eval_samples[time_index]
        eval_ambient = eval_sample[:,:2]
        eval_condition = eval_sample[:,2:]

        naive_samples = naive_forward(t0_ambient, t0_condition, eval_time, naive_model)

        wass_dists_model_i.append(compute_wasserstein_distance(eval_ambient, naive_samples))
    
    avg_wass_dists_naive.append(np.mean(wass_dists_model_i))
    print(f'Model {i}, average Wasserstein distance across eval times: {avg_wass_dists_naive[-1]}')

/tmp/ipykernel_6639/3696377404.py:7: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  naive_sample = naive_model(torch.tensor(sample, device=device), torch.tensor(lambda_val, device=device), torch.tensor(cond, device=device)).cpu().detach().numpy()


Model 0, average Wasserstein distance across eval times: 0.5161448381057708
Model 1, average Wasserstein distance across eval times: 0.5399456183586734
Model 2, average Wasserstein distance across eval times: 0.431855742702726
Model 3, average Wasserstein distance across eval times: 0.5032989428130681
Model 4, average Wasserstein distance across eval times: 0.39470431774278386
Model 5, average Wasserstein distance across eval times: 0.3643477511540986
Model 6, average Wasserstein distance across eval times: 0.4564073106871221
Model 7, average Wasserstein distance across eval times: 0.4921499868842073
Model 8, average Wasserstein distance across eval times: 0.4993275948935191
Model 9, average Wasserstein distance across eval times: 0.45598102782992617
Model 10, average Wasserstein distance across eval times: 0.37587904166552466
Model 11, average Wasserstein distance across eval times: 0.42452878901897917
Model 12, average Wasserstein distance across eval times: 0.499503845874977
Model 1

In [51]:
print_results(avg_wass_dists_naive, "Naive FiLM Model")

Naive FiLM Model: 0.458 ± 0.025


# CFM